# M0 · 07 — Masking & embedding lookup

Two more GPT-specific moves:
- **Boolean masking** with `masked_fill` — how the causal mask blocks the future.
- **Embedding lookup** with `nn.Embedding` — how token IDs become vectors.

Both *look* like loops but are single vectorised ops.

In [ ]:
import torch
from m0_checks import check, check_tensor, TODO
torch.manual_seed(0)

## 1. `torch.tril` — a lower-triangular matrix of ones

This is the raw material of the causal mask. `torch.tril(torch.ones(3,3))` keeps
the lower triangle (including diagonal) and zeros the rest.

Build the 3×3 lower-triangular ones matrix.

In [ ]:
tril = TODO
tril

In [ ]:
check('tril 3x3', tril, torch.tensor([[1., 0., 0.],
                                      [1., 1., 0.],
                                      [1., 1., 1.]]))

## 2. A boolean mask — where is the future?

`tril == 0` is `True` exactly at the **upper triangle** (future positions the
GPT must hide). Build that boolean mask.

In [ ]:
future = TODO
future

In [ ]:
check('future mask', future, torch.tensor([[False, True,  True],
                                           [False, False, True],
                                           [False, False, False]]))

## 3. `masked_fill` — overwrite masked positions

`scores.masked_fill(mask, value)` sets every `True` position to `value`. The GPT
uses `-inf` so that after softmax those spots become 0.

Fill the future positions of `scores` with `float('-inf')`.

In [ ]:
scores = torch.zeros(3, 3)
masked = TODO
masked

In [ ]:
from torch.nn import functional as F
# After softmax, the -inf entries become exactly 0 and each row sums to 1:
w = F.softmax(masked, dim=-1)
check('row 0 attends only to itself', w[0], torch.tensor([1.0, 0.0, 0.0]))
check('row 1 splits over first two', w[1], torch.tensor([0.5, 0.5, 0.0]))

## 4. Embedding = row lookup from a table

`nn.Embedding(num, dim)` is a `(num, dim)` table; calling it with IDs returns
those **rows**. It's mathematically `one_hot(ids) @ table`, done as a fast lookup.
This is `token_embedding(idx)` in the GPT.

Look up rows `[0, 2]` from the table below (rows 0 and 2).

In [ ]:
table = torch.nn.Embedding(4, 3)
with torch.no_grad():  # set known weights so the answer is predictable
    table.weight.copy_(torch.tensor([[0., 0., 0.],
                                      [1., 1., 1.],
                                      [2., 2., 2.],
                                      [3., 3., 3.]]))
ids = torch.tensor([0, 2])
rows = TODO
rows

In [ ]:
check('looked-up rows', rows, torch.tensor([[0., 0., 0.], [2., 2., 2.]]))

## 5. Lookup preserves the ID tensor's shape (+ embedding dim)

Give it `(B, T)` IDs and you get `(B, T, dim)` — exactly how `(B,T)` tokens become
`(B,T,C)` embeddings. Look up a `(2, 2)` batch of IDs; expect shape `(2, 2, 3)`.

In [ ]:
idx = torch.tensor([[0, 1], [2, 3]])  # (B=2, T=2)
emb = TODO
emb.shape

In [ ]:
check_tensor('embedded shape (2,2,3)', emb, shape=(2, 2, 3))

## ✅ Recap

- `torch.tril` + `mask == 0` build the causal 'future' mask.
- `masked_fill(mask, -inf)` hides positions; softmax then zeroes them.
- `nn.Embedding` is a learnable row-lookup: `(B,T)` IDs → `(B,T,C)` vectors.

Next: **08 — autograd & a tiny training loop** (how the model actually learns).